# 🥈 Transformação Bronze → Silver

Este notebook desenvolve a camada Silver da pipeline: lê os dados brutos da Bronze, aplica limpeza, padronização e decodificação, integra as bases entre si, incorpora os eventos do streaming como estimativa preliminar de 2025 e executa as regras de qualidade, com os registros reprovados isolados em quarentena. O resultado é gravado em `silver/`, a segunda área do medalhão no lake.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/transform/prod_03_bronze_to_silver.py`.

**Decisões que governam este notebook** (ver [diário de decisões](../docs/decisoes.md)):
- D-010: a rede Pública (código 5) é o recorte de referência das análises;
- D-011: todos os alunos são preservados com a flag `presente`; as regras de qualidade tratam nulos de forma condicional; a Gold publicará taxa oficial, taxa ajustada e participação;
- D-012: motor pandas, com as camadas mantidas como Parquet no lake;
- D-013: metas comparadas na safra vigente estrita, sem fallback (decisão nascida na Seção 5 deste notebook).

📚 **Referências das aulas (módulo Fase 2, Data Prepare):**
- *ETL Pipelines, Aula 1 (Pipelines ETL/ELT):* o papel da camada Silver (limpeza, padronização, validação e integração), a preservação de lineage e as dimensões de qualidade de dados;
- *Bancos de Dados Relacionais, Aula 1:* chaves e integridade referencial, aplicadas aqui nos joins de integração;
- *Notebook ETL_Pipeline_Demo_Parte1_Pandas:* implementação de referência da passagem Bronze → Silver vista em aula;
- Achados do levantamento das fontes ([dicionário de dados](../docs/dicionario_dados.md)): códigos da coluna `rede`, metas em formato wide e nulos estruturais dos ausentes.

---

**Estratégia de desenvolvimento deste notebook:**

```
decodificar   ->  padronizar e   ->  integrar   ->  estimativa      ->  qualidade e   ->  ensaio da
(laboratório)     transformar        as bases       2025 (fluxo)        quarentena        produção
Seção 2           Seções 3 e 4       Seção 5        Seção 6             Seção 7           Seção 8
```

Cada transformação é validada em tabela pequena antes de escalar, no mesmo princípio das etapas anteriores. Ao final, o ciclo completo é empacotado em funções, as unidades que serão promovidas para produção.

## 1. Setup do ambiente

**Passos desta seção:** ler a configuração do projeto, obter a credencial com o quota project declarado e definir os caminhos das camadas.

🎓 **Conceito** (*ETL Pipelines, Aula 1*): a Silver é a camada de curadoria: recebe o dado bruto da Bronze e entrega dado limpo, padronizado, integrado e validado. Nenhuma transformação acontece na Bronze (que permanece intacta como registro histórico); tudo o que esta etapa produz nasce em `silver/`.

> 📌 **Nota para reprodução:** este notebook lê o `config/config.json`, que não é versionado. Crie o seu a partir do `config/config.example.json`, como descrito na seção Como Executar do README.

In [3]:
# --- 1.1 Configuração, credencial e caminhos das camadas ---
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pydata_google_auth
from google.cloud import storage

# Configuração do executor (projeto e bucket próprios)
CFG = json.loads(Path("../config/config.json").read_text(encoding="utf-8"))
PROJETO_GCP = CFG["projeto_gcp"]
BUCKET_LAKE = CFG["bucket_lake"]

# Credencial ampla com o quota project declarado (lição da etapa de streaming)
ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]
credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
credenciais = credenciais.with_quota_project(PROJETO_GCP)

cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)

BRONZE = f"gs://{BUCKET_LAKE}/bronze"
SILVER = f"gs://{BUCKET_LAKE}/silver"

def ultima_particao(tabela: str) -> str:
    """Descobre a partição mais recente de uma tabela na Bronze."""
    particoes = sorted({
        b.name.split("/")[2]
        for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix=f"bronze/{tabela}/")
    })
    return particoes[-1]

def garantir_credencial() -> None:
    """Renova o token se expirado e limpa o cache do gcsfs.

    Tokens de acesso expiram em cerca de uma hora; em sessões longas de
    notebook, o gcsfs pode reter uma instância com token vencido (erro 401
    ou leitura travada). A verificação antes de cada acesso previne isso.
    """
    import google.auth.transport.requests
    import gcsfs
    if not credenciais.valid:
        credenciais.refresh(google.auth.transport.requests.Request())
        gcsfs.GCSFileSystem.clear_instance_cache()

def ler_bronze(tabela: str, **kwargs) -> pd.DataFrame:
    """Lê a partição mais recente de uma tabela batch da Bronze."""
    garantir_credencial()
    caminho = f"{BRONZE}/{tabela}/{ultima_particao(tabela)}/{tabela}.parquet"
    return pd.read_parquet(caminho, storage_options={"token": credenciais}, **kwargs)

print(f"Origem:  {BRONZE}/")
print(f"Destino: {SILVER}/")
print("Setup ok")

Origem:  gs://tech-challenge-fase2-lake-rm373453/bronze/
Destino: gs://tech-challenge-fase2-lake-rm373453/silver/
Setup ok


## 2. Decodificação: os códigos da fonte ganham significado

**Passos desta seção:** (2.1) carregar a tabela `dicionario` da Bronze e transformá-la em mapas de decodificação; (2.2) aplicar a decodificação na tabela `uf`, o laboratório barato, conferindo o resultado.

🎓 **Conceito, dado codificado e tabela de domínio** (*Bancos de Dados Relacionais, Aula 1*): a fonte armazena códigos (`rede = "3"`) e mantém uma tabela separada com os significados (`3 = Municipal`). É o padrão de tabela de domínio dos bancos relacionais. Na Silver, o código é preservado (é a chave) e o significado entra como coluna nova (`rede_nome`), tornando a camada legível sem consulta ao dicionário.

🎓 **Conceito, o dicionário como dado, não como documento:** o levantamento das fontes mostrou que os significados vivem na tabela `dicionario` (27 linhas). Em vez de escrever os mapas manualmente no código (que dessincronizariam se a fonte mudasse), a decodificação é construída a partir da própria tabela, na Bronze.

In [4]:
# --- 2.1 Construir os mapas de decodificação a partir do dicionário ---
df_dicionario = ler_bronze("dicionario")

def mapa_decodificacao(id_tabela: str, coluna: str) -> dict:
    """Extrai do dicionário o mapa codigo -> significado de uma coluna."""
    filtro = (df_dicionario["id_tabela"] == id_tabela) &              (df_dicionario["nome_coluna"] == coluna)
    return dict(zip(df_dicionario.loc[filtro, "chave"],
                    df_dicionario.loc[filtro, "valor"]))

MAPA_REDE_AGREGADAS = mapa_decodificacao("uf", "rede")       # tabelas uf e municipio
MAPA_REDE_ALUNOS    = mapa_decodificacao("alunos", "rede")   # microdados (códigos 1 a 4)

print("Mapa de rede (tabelas agregadas):")
for codigo, nome in sorted(MAPA_REDE_AGREGADAS.items()):
    print(f"  {codigo} -> {nome}")

Mapa de rede (tabelas agregadas):
  0 -> Total (Federal, Estadual, Municipal e Privada)
  1 -> Federal
  2 -> Estadual
  3 -> Municipal
  4 -> Privada
  5 -> Pública (Estadual e Municipal)
  6 -> Pública (Federal, Estadual e Municipal)


In [5]:
# --- 2.2 Laboratório barato: decodificar a tabela uf e conferir ---
df_uf = ler_bronze("uf")

# O código permanece (é a chave); o significado entra como coluna nova
df_uf["rede_nome"] = df_uf["rede"].map(MAPA_REDE_AGREGADAS)

# Conferência 1: algum código ficou sem tradução?
sem_traducao = df_uf["rede_nome"].isna().sum()
print(f"Linhas sem tradução de rede: {sem_traducao}  (esperado: 0)")

# Conferência 2: a distribuição decodificada
print()
print(df_uf.groupby(["rede", "rede_nome"], observed=True).size()
      .rename("linhas").reset_index().to_string(index=False))

Linhas sem tradução de rede: 0  (esperado: 0)

rede                                      rede_nome  linhas
   0 Total (Federal, Estadual, Municipal e Privada)       1
   2                                       Estadual      46
   3                                      Municipal      49
   5                 Pública (Estadual e Municipal)      49


## 3. Unpivot das metas: de colunas para linhas

**Passos desta seção:** (3.1) demonstrar a transformação na menor tabela de metas (`meta_alfabetizacao_brasil`, 3 linhas), comparando o antes e o depois; (3.2) empacotar em função e aplicar às três tabelas de metas, conferindo as contagens.

🎓 **Conceito, wide × long** (achado do levantamento; *ETL Pipelines, Aula 1*): a fonte entrega as metas em formato **wide**: uma coluna por ano (`meta_alfabetizacao_2024` ... `meta_alfabetizacao_2030`). Esse formato dificulta a pergunta central da Gold ("a taxa de 2025 atingiu a meta de 2025?"), que exige casar resultado e meta **pelo ano**. O formato **long** resolve: cada meta vira uma linha com `ano_meta` e `meta_taxa`, e o casamento vira um join simples. No pandas, a operação é o `melt`.

⚠️ **Nuance descoberta nos dados:** cada linha da fonte carrega o conjunto completo de metas, e as linhas de anos diferentes repetem essas metas (com pequenas revisões: a safra de 2025 traz valores levemente ajustados). No formato long, o `ano` original é preservado como `ano_referencia` (a safra da publicação); a escolha da safra mais recente acontece na integração, adiante.

In [6]:
# --- 3.1 Laboratório barato: o unpivot na tabela Brasil (3 linhas) ---
df_meta_br = ler_bronze("meta_alfabetizacao_brasil")

colunas_meta = [c for c in df_meta_br.columns if c.startswith("meta_alfabetizacao_2")]
print(f"ANTES (wide): {len(df_meta_br)} linhas, com {len(colunas_meta)} colunas de meta")
print(df_meta_br[["ano", "rede", "taxa_alfabetizacao"] + colunas_meta[:3]].to_string(index=False))
print()

df_meta_br_long = df_meta_br.melt(
    id_vars=["ano", "rede"],              # chaves preservadas
    value_vars=colunas_meta,              # colunas que viram linhas
    var_name="ano_meta",
    value_name="meta_taxa",
)
# 'meta_alfabetizacao_2024' -> 2024 (inteiro)
df_meta_br_long["ano_meta"] = (df_meta_br_long["ano_meta"]
                               .str.replace("meta_alfabetizacao_", "")
                               .astype(int))
df_meta_br_long = df_meta_br_long.rename(columns={"ano": "ano_referencia"})

print(f"DEPOIS (long): {len(df_meta_br_long)} linhas  "
      f"({len(df_meta_br)} x {len(colunas_meta)} metas)")
df_meta_br_long.head(8)

ANTES (wide): 3 linhas, com 7 colunas de meta
 ano    rede  taxa_alfabetizacao  meta_alfabetizacao_2024  meta_alfabetizacao_2025  meta_alfabetizacao_2026
2025 Pública                66.0                     60.0                    64.00                    67.00
2024 Pública                59.2                     59.9                    63.77                    67.47
2023 Pública                55.9                     59.9                    63.77                    67.47

DEPOIS (long): 21 linhas  (3 x 7 metas)


,ano_referencia,rede,ano_meta,meta_taxa
0,2025,Pública,2024,60.00
1,2024,Pública,2024,59.90
2,2023,Pública,2024,59.90
3,2025,Pública,2025,64.00
4,2024,Pública,2025,63.77
5,2023,Pública,2025,63.77
6,2025,Pública,2026,67.00
7,2024,Pública,2026,67.47


In [7]:
# --- 3.2 Empacotar em função e aplicar às três tabelas de metas ---
def unpivot_metas(df: pd.DataFrame, chaves: list) -> pd.DataFrame:
    """Converte as colunas meta_alfabetizacao_AAAA em linhas (formato long)."""
    colunas = [c for c in df.columns if c.startswith("meta_alfabetizacao_2")]
    longo = df.melt(id_vars=chaves, value_vars=colunas,
                    var_name="ano_meta", value_name="meta_taxa")
    longo["ano_meta"] = (longo["ano_meta"]
                         .str.replace("meta_alfabetizacao_", "")
                         .astype(int))
    return longo.rename(columns={"ano": "ano_referencia"})

metas_long = {
    "brasil":    unpivot_metas(ler_bronze("meta_alfabetizacao_brasil"),
                               ["ano", "rede"]),
    "uf":        unpivot_metas(ler_bronze("meta_alfabetizacao_uf"),
                               ["ano", "sigla_uf", "rede"]),
    "municipio": unpivot_metas(ler_bronze("meta_alfabetizacao_municipio"),
                               ["ano", "id_municipio", "rede"]),
}

for nome, df_long in metas_long.items():
    print(f"  metas_{nome:<10} {len(df_long):>8,} linhas no formato long")

  metas_brasil           21 linhas no formato long
  metas_uf              567 linhas no formato long
  metas_municipio    74,928 linhas no formato long


## 4. Alunos: a flag de presença

**Passos desta seção:** (4.1) carregar os microdados de alunos da Bronze, selecionando apenas as colunas com utilidade analítica, e criar a flag `presente`; (4.2) validar nos dados as premissas das regras condicionais de qualidade da D-011.

🎓 **Conceito, nulo estrutural vira dado explícito** (decisão D-011; *ETL Pipelines, Aula 1, dimensão completude*): os alunos ausentes têm proficiência e peso nulos por construção da fonte, não por defeito. A Silver torna essa informação explícita com a flag booleana `presente`: nenhuma linha é descartada, e qualquer análise escolhe seu recorte com clareza. As agregações oficiais da Gold usarão os presentes ponderados (método INEP); a taxa ajustada usará todos.

ℹ️ **Colunas descartadas na seleção:** `id_escola` (máscara fictícia, sem valor de cruzamento, conforme o levantamento), `caderno` (identificação da prova) e `preenchimento_caderno` e `serie` (constantes ou redundantes com `presenca`). A Bronze permanece com tudo; a Silver carrega o que serve à análise.

In [8]:
# --- 4.1 Carregar alunos e criar a flag de presença ---
df_alunos = ler_bronze(
    "alunos",
    columns=["ano", "id_municipio", "id_aluno", "rede",
             "presenca", "alfabetizado", "proficiencia", "peso_aluno"],
)

# A flag da D-011: presença explícita como booleano
df_alunos["presente"] = df_alunos["presenca"] == "1"

# Decodificação da rede (mapa da Seção 2, códigos 1 a 4 dos microdados)
df_alunos["rede_nome"] = df_alunos["rede"].map(MAPA_REDE_ALUNOS)

print(f"Alunos carregados: {len(df_alunos):,}")
print()
print(df_alunos.groupby(["ano", "presente"]).size()
      .rename("alunos").reset_index().to_string(index=False))

Alunos carregados: 3,867,999

 ano  presente  alunos
2023     False  244381
2023      True 1503058
2024     False  267772
2024      True 1852788


In [9]:
# --- 4.2 Validar as premissas das regras condicionais (D-011) ---
# Regra: proficiência nula é legítima em AUSENTE; anômala em PRESENTE.
# Antes de codificar a regra na Seção de qualidade, conferimos se os
# dados sustentam a premissa.

presente_sem_nota = ((df_alunos["presente"]) &
                     (df_alunos["proficiencia"].isna())).sum()
ausente_com_nota  = ((~df_alunos["presente"]) &
                     (df_alunos["proficiencia"].notna())).sum()
ausente_sem_nota  = ((~df_alunos["presente"]) &
                     (df_alunos["proficiencia"].isna())).sum()

print("Cruzamento presença × proficiência:")
print(f"  presente COM nota (esperado: maioria):   "
      f"{(df_alunos['presente'] & df_alunos['proficiencia'].notna()).sum():,}")
print(f"  ausente SEM nota (legítimo, D-011):      {ausente_sem_nota:,}")
print(f"  presente SEM nota (anomalia esperada 0): {presente_sem_nota:,}")
print(f"  ausente COM nota (anomalia esperada 0):  {ausente_com_nota:,}")
print()

# O peso acompanha a proficiência? (premissa da agregação ponderada)
peso_nulo_diferente = (df_alunos["peso_aluno"].isna() !=
                       df_alunos["proficiencia"].isna()).sum()
print(f"Linhas onde peso e proficiência divergem em nulidade: {peso_nulo_diferente:,}"
      f"  (esperado: 0)")

Cruzamento presença × proficiência:
  presente COM nota (esperado: maioria):   3,354,661
  ausente SEM nota (legítimo, D-011):      512,153
  presente SEM nota (anomalia esperada 0): 1,185
  ausente COM nota (anomalia esperada 0):  0

Linhas onde peso e proficiência divergem em nulidade: 0  (esperado: 0)


### 4.3 A validação encontrou um caso real

⚠️ A conferência da célula 4.2 mostrou que a anomalia **presente sem nota** existe na fonte: **1.185 registros** (0,03% do total). A premissa de zero não se confirmou, e é para isso que a validação serve: a regra de qualidade da Seção 6 vai tratar exatamente esses registros, destinando-os à quarentena, como a D-011 previu. As demais premissas se confirmaram (nenhum ausente com nota; peso e proficiência sempre nulos juntos, o que protege a agregação ponderada).

Antes de codificar a regra, a célula abaixo caracteriza os registros anômalos: de que anos e redes vêm, e o que o campo `alfabetizado` diz sobre eles.

In [10]:
# --- 4.3 Investigar a anomalia: presentes sem nota ---
anomalos = df_alunos[df_alunos["presente"] & df_alunos["proficiencia"].isna()]

print(f"Registros anômalos: {len(anomalos):,}  "
      f"({len(anomalos) / len(df_alunos):.3%} do total)")
print()
print("Distribuição por ano e rede:")
print(anomalos.groupby(["ano", "rede_nome"], observed=True).size()
      .rename("alunos").reset_index().to_string(index=False))
print()
print("Campo alfabetizado nesses registros:")
print(anomalos["alfabetizado"].value_counts(dropna=False).to_string())
print()
print(f"Municípios afetados: {anomalos['id_municipio'].nunique():,}")

Registros anômalos: 1,185  (0.031% do total)

Distribuição por ano e rede:
 ano rede_nome  alunos
2023  Estadual       4
2023 Municipal     245
2024  Estadual      99
2024 Municipal     837

Campo alfabetizado nesses registros:
alfabetizado
0    1185

Municípios afetados: 244


## 5. Integração: as bases se encontram

**Passos desta seção:** (5.1) carregar o diretório de municípios da Bronze; (5.2) integrar os resultados municipais com o diretório, executando as conferências que protegem um join; (5.3) selecionar a safra vigente das metas e integrá-la aos resultados.

🎓 **Conceito, fato e dimensão** (*Bancos de Dados Relacionais, Aula 1*): as tabelas de resultado carregam chaves (`id_municipio`) e medidas (taxas); quem descreve as entidades por trás das chaves são as tabelas de dimensão. O diretório de municípios da Base dos Dados cumpre esse papel: para cada `id_municipio` (código IBGE de 7 dígitos, a chave universal de município no Brasil), traz nome, UF e região. Sem ele, a Silver entregaria tabelas que só um computador lê.

📌 **Nota de evolução do escopo:** ao preparar a integração, a verificação das tabelas disponíveis mostrou que as 7 bases ingeridas do INEP são todas tabelas de **fato**: nenhuma carrega os atributos descritivos do município, e um `distinct` na tabela de resultados devolveria apenas a lista de códigos (e apenas dos municípios avaliados). Foi necessário acrescentar uma tabela de **dimensão** ao lake. Seguindo o princípio de que todo dado externo entra pela ingestão, a tabela `diretorio_municipio` (fonte `br_bd_diretorios_brasil.municipio`) foi adicionada ao `prod_01_ingestao_batch.py`, e este notebook a consome da Bronze, como qualquer outra. O `desenv_01` registra essa evolução em nota ao final.

🎓 **Conceito, as conferências de um join** (*Bancos de Dados Relacionais, Aula 1, integridade referencial*): todo join da célula 5.2 em diante é acompanhado de duas verificações. **Contagem de linhas**: um join muitos-para-um não pode criar nem eliminar linhas; se a contagem mudar, há chave duplicada na dimensão (explosão) ou erro de tipo na chave. **Correspondência completa**: com `how="left"`, chave sem par na dimensão vira nulo, e contamos esses nulos (esperado: zero).

In [12]:
# --- 5.1 Carregar o diretório de municípios da Bronze ---
df_diretorio = ler_bronze(
    "diretorio_municipio",
    columns=["id_municipio", "nome", "sigla_uf", "nome_uf", "nome_regiao"],
)

# Conferência: a chave da dimensão precisa ser única (protege o join da 5.2)
duplicadas = df_diretorio["id_municipio"].duplicated().sum()

print(f"Municípios no diretório: {len(df_diretorio):,}")
print(f"Chaves duplicadas: {duplicadas}  (esperado: 0)")
df_diretorio.head(3)

Municípios no diretório: 5,571
Chaves duplicadas: 0  (esperado: 0)


,id_municipio,nome,sigla_uf,nome_uf,nome_regiao
0,5101837,Boa Esperança do Norte,MT,Mato Grosso,Centro-Oeste
1,1100205,Porto Velho,RO,Rondônia,Norte
2,1100338,Nova Mamoré,RO,Rondônia,Norte


In [13]:
# --- 5.2 Integrar os resultados municipais com o diretório ---
df_municipio = ler_bronze("municipio")
df_municipio["rede_nome"] = df_municipio["rede"].map(MAPA_REDE_AGREGADAS)

antes = len(df_municipio)
df_municipio_int = df_municipio.merge(df_diretorio, on="id_municipio", how="left")

# Conferência 1: join muitos-para-um não cria nem elimina linhas
print(f"Linhas antes do join:  {antes:,}")
print(f"Linhas depois do join: {len(df_municipio_int):,}  (esperado: igual)")

# Conferência 2: todo município avaliado existe no diretório?
sem_nome = df_municipio_int["nome"].isna().sum()
print(f"Municípios sem correspondência no diretório: {sem_nome}  (esperado: 0)")

# Amostra legível: a razão de ser da integração
print()
colunas_amostra = [c for c in ["ano", "id_municipio", "nome", "sigla_uf",
                               "nome_regiao", "rede_nome", "taxa_alfabetizacao"]
                   if c in df_municipio_int.columns]
print(df_municipio_int[colunas_amostra].head(5).to_string(index=False))

Linhas antes do join:  23,995
Linhas depois do join: 23,995  (esperado: igual)
Municípios sem correspondência no diretório: 0  (esperado: 0)

 ano id_municipio            nome sigla_uf nome_regiao                      rede_nome  taxa_alfabetizacao
2023      1100031          Cabixi       RO       Norte                      Municipal               69.10
2023      1100072      Corumbiara       RO       Norte                      Municipal               58.20
2023      1100189   Pimenta Bueno       RO       Norte Pública (Estadual e Municipal)               69.73
2023      1101609       Theobroma       RO       Norte                      Municipal               50.70
2023      1101807 Vale do Paraíso       RO       Norte                      Municipal               55.69


### 5.3 A safra vigente das metas

Três regras governam esta célula, todas decididas antes dela existir:

- **Seleção da safra** (nuance da Seção 3): as metas existem em três versões (`ano_referencia` 2023, 2024 e 2025), com valores renegociados entre elas. A versão que vale é a da safra mais recente: `ano_referencia` máximo.
- **Aplicação por rede** (decisão D-010): as metas do Compromisso são pactuadas para a rede pública. Nas demais redes, `meta_taxa` fica nula por definição, e não por falha de join.
- **Nulo legítimo no tempo:** os resultados de 2023 não recebem meta, porque as metas pactuadas começam em 2024. É o mesmo padrão dos ausentes sem proficiência: a ausência carrega significado, e a conferência ao final mostra essa fronteira nos números.

In [14]:
# --- 5.3 Selecionar a safra vigente das metas e integrar ---
metas_mun = metas_long["municipio"]

safra_vigente = metas_mun["ano_referencia"].max()
metas_vigentes = metas_mun.loc[
    metas_mun["ano_referencia"] == safra_vigente,
    ["id_municipio", "ano_meta", "meta_taxa"],
]

print(f"Safra vigente: {safra_vigente}")
print(f"Metas vigentes: {len(metas_vigentes):,} linhas  "
      f"({metas_vigentes['id_municipio'].nunique():,} municípios, "
      f"anos {metas_vigentes['ano_meta'].min()}-{metas_vigentes['ano_meta'].max()})")
print()

# Integração: a meta do ano X encontra o resultado do ano X
antes = len(df_municipio_int)
df_municipio_silver = df_municipio_int.merge(
    metas_vigentes,
    left_on=["id_municipio", "ano"],
    right_on=["id_municipio", "ano_meta"],
    how="left",
).drop(columns="ano_meta")

# D-010: a meta vale para a rede pública; nas demais, nula por definição
df_municipio_silver.loc[df_municipio_silver["rede"].astype(str) != "5",
                        "meta_taxa"] = pd.NA

print(f"Linhas antes do join:  {antes:,}")
print(f"Linhas depois do join: {len(df_municipio_silver):,}  (esperado: igual)")
print()

# Conferência: cobertura de meta na rede Pública, por ano do resultado
rede5 = df_municipio_silver[df_municipio_silver["rede"].astype(str) == "5"]
print("Cobertura de meta na rede Pública (código 5):")
print(rede5.groupby("ano")["meta_taxa"]
      .agg(linhas="size", com_meta="count").to_string())
print()
print("Leitura esperada: 2023 sem meta (pactuação começa em 2024);")
print("2024 com meta na quase totalidade dos municípios.")

Safra vigente: 2024
Metas vigentes: 37,464 linhas  (5,352 municípios, anos 2024-2030)

Linhas antes do join:  23,995
Linhas depois do join: 23,995  (esperado: igual)

Cobertura de meta na rede Pública (código 5):
      linhas  com_meta
ano                   
2023    4950         0
2024    5516      5232

Leitura esperada: 2023 sem meta (pactuação começa em 2024);
2024 com meta na quase totalidade dos municípios.


### 5.4 O que os números da integração revelaram

Dois resultados divergiram da expectativa, e ambos deixam lição:

- ⚠️ **A safra vigente municipal é 2024, não 2025.** A tabela Brasil tem três safras (2023 a 2025), mas a municipal tem duas (2023 e 2024): a renegociação de 2025 ainda não foi publicada no grão municipal. Como a seleção usa o máximo disponível (`ano_referencia.max()`) em vez de um ano fixo, a regra permanece correta e passará a usar a safra nova automaticamente quando a fonte a publicar.
- ⚠️ **284 municípios têm resultado em 2024 mas não têm meta** (5.516 avaliados na rede pública contra 5.232 com meta). A tabela de metas cobre 5.352 municípios, mais do que os avaliados, e ainda assim esses 284 ficaram de fora. A célula abaixo os caracteriza antes de qualquer decisão de tratamento.

In [15]:
# --- 5.4 Investigar: resultados de 2024 sem meta pactuada ---
sem_meta = rede5[(rede5["ano"] == 2024) & (rede5["meta_taxa"].isna())]
print(f"Municípios com resultado 2024 e sem meta: {len(sem_meta)}")

# Eles aparecem em alguma safra da tabela de metas?
em_metas = sem_meta["id_municipio"].isin(metas_mun["id_municipio"])
print(f"Presentes na tabela de metas em alguma safra: {em_metas.sum()}  "
      f"(0 = ausência total de pactuação na fonte)")
print()

print("Distribuição por UF (top 8):")
print(sem_meta["sigla_uf"].value_counts().head(8).to_string())
print()
print("Amostra:")
print(sem_meta[["id_municipio", "nome", "sigla_uf", "taxa_alfabetizacao"]]
      .head(5).to_string(index=False))

Municípios com resultado 2024 e sem meta: 284
Presentes na tabela de metas em alguma safra: 120  (0 = ausência total de pactuação na fonte)

Distribuição por UF (top 8):
sigla_uf
RS    57
SC    54
MG    51
RN    28
AC    22
SP    21
AM    14
BA    13

Amostra:
id_municipio                  nome sigla_uf  taxa_alfabetizacao
     2406155                Jundiá       RN               33.33
     4315321              Quevedos       RS               25.00
     4309126 Gramado dos Loureiros       RS               37.50
     4301750      Barão do Triunfo       RS                4.40
     2414456      Triunfo Potiguar       RN               10.95


### 5.5 Decisão registrada: D-013, safra vigente estrita

Com a caracterização da célula 5.4, a decisão ficou registrada no [diário de decisões](../docs/decisoes.md) como **D-013**: a comparação meta × resultado usa exclusivamente a safra vigente (`ano_referencia` máximo), sem fallback para safras anteriores. Os 284 municípios avaliados sem meta permanecem com `meta_taxa` nula, na categoria documentada **sem meta na safra vigente** (164 sem pactuação em qualquer safra; 120 fora da repactuação de 2024). O racional: um município que saiu da repactuação mais recente pode ter tido a meta revista, e usar a publicação antiga apresentaria inferência como fato; o custo declarado é deixar cerca de 5% dos municípios avaliados fora da comparação com meta na Gold.

Com isso, a Seção 5 entrega o `df_municipio_silver`: resultados municipais decodificados, com nome, UF e região do diretório e a meta da safra vigente. A Seção 6 cuida das regras de qualidade e da quarentena, onde este e os demais nulos legítimos serão formalizados.

## 6. Estimativa 2025: o fluxo vira indicador

**Passos desta seção:** (6.1) ler e empilhar os micro-lotes de eventos gravados pela ingestão streaming na Bronze; (6.2) derivar a classificação de alfabetização e agregar de forma ponderada por município; (6.3) materializar a estimativa com origem explícita e confrontá-la com a meta pactuada para 2025.

🎓 **Conceito, micro-lotes viram tabela lógica** (*ETL Pipelines, Aula 2, ingestão streaming; notebook Kafka_Demo*): o consumidor do fluxo grava um arquivo Parquet por micro-lote (`bronze/eventos_resultado_aluno/data_processamento=.../lote_HHMMSS.parquet`). Individualmente são fotografias de minutos diferentes; empilhados, formam a tabela lógica de eventos. A leitura lista todos os arquivos do prefixo e os concatena, e a deduplicação por `id_evento` feita pelo consumidor é verificada aqui como conferência.

🎓 **Conceito, estimativa não é dado oficial** (decisão D-005): os eventos simulam o ciclo de 2025 amostrando a distribuição real dos presentes de 2024. A agregação produz uma **estimativa preliminar** do indicador, identificada como tal na coluna de origem, válida apenas onde o consolidado oficial ainda não existe e descartada quando ele chegar.

📌 **Derivação da classificação:** o contrato de evento não carrega o campo `alfabetizado` de propósito (dado derivável não trafega). A classificação nasce aqui: proficiência maior ou igual a **743**, o limite inferior do nível considerado alfabetizado, verificado empiricamente no levantamento das fontes.

🎓 **Conceito, composição de estoque e fluxo (arquitetura Lambda):** o indicador servido pela pipeline compõe as duas naturezas de dado **no eixo do tempo**: o histórico consolidado do batch é o estoque da série (2023 e 2024) e a estimativa do fluxo move o ponteiro do ponto corrente (2025). A junção acontece na série histórica, nunca dentro do mesmo ponto: cada ciclo avaliativo mede uma coorte nova de alunos e é autocontido (a taxa de 2025 não é "2024 mais um delta", como seria um saldo bancário). O ciclo em andamento é, ele próprio, um estoque que cresce: a leitura da célula 6.1 empilha todos os micro-lotes acumulados, e cada novo consumo do fluxo move a estimativa no recálculo. Quando o consolidado oficial do ciclo for publicado, o ponto estimado é substituído e passa a integrar o estoque (decisão D-005).

In [16]:
# --- 6.1 Ler e empilhar os micro-lotes do fluxo ---
def ler_micro_lotes(tabela: str) -> pd.DataFrame:
    """Lê todos os micro-lotes de uma tabela de eventos e os empilha."""
    garantir_credencial()
    blobs = [b.name for b in
             cliente_storage.list_blobs(BUCKET_LAKE, prefix=f"bronze/{tabela}/")]
    lotes = [pd.read_parquet(f"gs://{BUCKET_LAKE}/{caminho}",
                             storage_options={"token": credenciais})
             for caminho in blobs]
    return pd.concat(lotes, ignore_index=True), len(blobs)

df_eventos, qtd_lotes = ler_micro_lotes("eventos_resultado_aluno")

print(f"Micro-lotes lidos: {qtd_lotes}")
print(f"Eventos empilhados: {len(df_eventos):,}")

# Conferência: a deduplicação do consumidor garantiu id_evento único?
duplicados = df_eventos["id_evento"].duplicated().sum()
print(f"id_evento duplicado: {duplicados}  (esperado: 0, dedup do consumidor)")
print()
print(f"Municípios distintos no fluxo: {df_eventos['id_municipio'].nunique():,}")
df_eventos.head(3)

Micro-lotes lidos: 4
Eventos empilhados: 2,222
id_evento duplicado: 0  (esperado: 0, dedup do consumidor)

Municípios distintos no fluxo: 1,163


,id_evento,schema_version,event_time,id_municipio,rede,id_aluno,proficiencia,peso_aluno,_processing_timestamp,_source,data_processamento
0,e5f5687a-c03e-476a-b261-4025db412967,1,2026-07-10T19:10:18.840558+00:00,3201506,3,A2025-583304,807.0,1.1100,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
1,b77f6d64-148a-41b1-bb32-94504619315b,1,2026-07-10T19:23:16.649907+00:00,3201506,3,A2025-612760,807.0,1.1100,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10
2,edc6b632-8005-4d5a-bec3-e7c357aeb032,1,2026-07-10T19:10:19.124770+00:00,3552809,3,A2025-283077,694.3,1.2353,2026-07-10T19:36:11.652160+00:00,pubsub:eventos-resultado-aluno,2026-07-10


In [17]:
# --- 6.2 Derivar a classificação e agregar ponderado por município ---
CORTE_ALFABETIZACAO = 743.0   # limite inferior do nível alfabetizado (desenv_00)

df_eventos["alfabetizado"] = df_eventos["proficiencia"] >= CORTE_ALFABETIZACAO
df_eventos["rede_nome"] = df_eventos["rede"].map(MAPA_REDE_ALUNOS)

print("Eventos por rede:")
print(df_eventos["rede_nome"].value_counts().to_string())
print()

# Recorte comparável à rede Pública código 5 (D-010): Estadual + Municipal
publica = df_eventos[df_eventos["rede_nome"].isin(["Estadual", "Municipal"])].copy()
print(f"Eventos no recorte da rede pública: {len(publica):,}")
print()

# Agregação ponderada: a mesma fórmula da taxa oficial do INEP
publica["peso_alfabetizado"] = publica["peso_aluno"] * publica["alfabetizado"]
df_estimativa = (publica.groupby("id_municipio")
                 .agg(alunos_no_fluxo=("id_evento", "size"),
                      soma_pesos=("peso_aluno", "sum"),
                      soma_pesos_alfabetizados=("peso_alfabetizado", "sum"))
                 .reset_index())
df_estimativa["taxa_estimada"] = (100 * df_estimativa["soma_pesos_alfabetizados"]
                                  / df_estimativa["soma_pesos"])

taxa_geral = 100 * publica["peso_alfabetizado"].sum() / publica["peso_aluno"].sum()
print(f"Municípios com estimativa: {len(df_estimativa):,}")
print(f"Taxa ponderada geral do fluxo: {taxa_geral:.1f}%")
print("(sanidade: os eventos amostram a distribuição de 2024, taxa oficial 59,2%)")
print()
print(df_estimativa.sort_values("alunos_no_fluxo", ascending=False)
      .head(5).to_string(index=False))

Eventos por rede:
rede_nome
Municipal    1931
Estadual      291

Eventos no recorte da rede pública: 2,222

Municípios com estimativa: 1,163
Taxa ponderada geral do fluxo: 60.0%
(sanidade: os eventos amostram a distribuição de 2024, taxa oficial 59,2%)

id_municipio  alunos_no_fluxo  soma_pesos  soma_pesos_alfabetizados  taxa_estimada
     3550308              104    121.0546                    66.315      54.781066
     3304557               49     55.9200                    35.300      63.125894
     3106200               26     31.3400                    23.210      74.058711
     2304400               23     23.3900                    19.360      82.770415
     5300108               23     27.5100                    16.640      60.487096


### 6.3 Materialização: a estimativa ganha identidade

Três cuidados definem esta célula:

- **Origem explícita** (decisão D-005): a coluna `origem = "estimativa_streaming"` acompanha cada linha. A estimativa nunca se disfarça de dado oficial; quando o consolidado de 2025 for publicado, as linhas desta origem são descartadas.
- **Meta para 2025 na safra vigente:** a D-013 mostrou que a repactuação municipal de 2025 ainda não existe, mas a meta *para* 2025 existe, pactuada na safra 2024. É a distinção entre `ano_meta` e `ano_referencia` (Seção 3) sustentando a comparação.
- **Fragilidade amostral declarada:** com cerca de 2 eventos por município em média, a taxa municipal individual é instável (1 evento produz taxa 0% ou 100%). A coluna `alunos_no_fluxo` viaja com o dado, e o corte mínimo de volume é decisão de quem consome; aqui, um corte de demonstração responde a primeira pergunta de negócio da pipeline.

In [18]:
# --- 6.3 Materializar a estimativa com origem explícita e meta 2025 ---
df_estimativa_2025 = df_estimativa[["id_municipio", "taxa_estimada",
                                    "alunos_no_fluxo"]].copy()
df_estimativa_2025.insert(0, "ano", 2025)
df_estimativa_2025["origem"] = "estimativa_streaming"

# Dimensão da Seção 5: nome, UF e região
df_estimativa_2025 = df_estimativa_2025.merge(
    df_diretorio, on="id_municipio", how="left")

# Meta pactuada PARA 2025, na safra vigente (D-013)
metas_2025 = metas_vigentes.loc[metas_vigentes["ano_meta"] == 2025,
                                ["id_municipio", "meta_taxa"]]
antes = len(df_estimativa_2025)
df_estimativa_2025 = df_estimativa_2025.merge(
    metas_2025, on="id_municipio", how="left")

print(f"Linhas antes/depois dos joins: {antes:,} / {len(df_estimativa_2025):,}"
      f"  (esperado: igual)")
print(f"Municípios com meta para 2025: "
      f"{df_estimativa_2025['meta_taxa'].notna().sum():,}")
print()

# Primeira leitura de negócio: rumo da meta segundo o fluxo
MINIMO_EVENTOS = 20   # corte de demonstração; o definitivo é decisão da Gold
robustos = df_estimativa_2025[
    (df_estimativa_2025["alunos_no_fluxo"] >= MINIMO_EVENTOS)
    & df_estimativa_2025["meta_taxa"].notna()
].copy()
robustos["no_rumo"] = robustos["taxa_estimada"] >= robustos["meta_taxa"]

print(f"Municípios com pelo menos {MINIMO_EVENTOS} eventos e meta: {len(robustos)}")
print(f"No rumo da meta de 2025 (estimativa >= meta): {robustos['no_rumo'].sum()}")
print()
print(robustos.sort_values("alunos_no_fluxo", ascending=False)
      [["nome", "sigla_uf", "alunos_no_fluxo", "taxa_estimada",
        "meta_taxa", "no_rumo"]]
      .head(8).to_string(index=False))

Linhas antes/depois dos joins: 1,163 / 1,163  (esperado: igual)
Municípios com meta para 2025: 1,140

Municípios com pelo menos 20 eventos e meta: 4
No rumo da meta de 2025 (estimativa >= meta): 3

          nome sigla_uf  alunos_no_fluxo  taxa_estimada  meta_taxa  no_rumo
     São Paulo       SP              104      54.781066      51.06     True
Rio de Janeiro       RJ               49      63.125894      63.97    False
Belo Horizonte       MG               26      74.058711      64.91     True
     Fortaleza       CE               23      82.770415      75.85     True


## 7. Qualidade e quarentena: as regras viram código

**Passos desta seção:** (7.1) aplicar aos microdados de alunos as regras estruturais da D-011, separando os reprovados em quarentena com o motivo carimbado; (7.2) auditar os nulos legítimos que permanecem na Silver com sua semântica preservada.

🎓 **Conceito, qualidade como regra executável** (*ETL Pipelines, Aula 1, dimensões de qualidade*): completude, validade e consistência deixam de ser aspirações e viram condições verificáveis em código. Cada regra desta seção foi validada empiricamente antes de existir (células 4.2 e 4.3): a regra aponta para casos que sabemos que existem, na quantidade que sabemos esperar.

🎓 **Conceito, quarentena preserva, não apaga:** o registro reprovado não é descartado; é isolado em área própria do lake com o motivo da reprovação. Isso mantém a pipeline auditável (é possível responder "por que este aluno não está na Silver?") e reprocessável (se a regra mudar, a quarentena é a fila de reavaliação).

📌 **O mapa completo da qualidade na pipeline.** Cada tipo de problema tem seu tribunal, e eles não se confundem:

| Situação | Onde é detectada | Destino |
|---|---|---|
| Evento malformado ou duplicado no fluxo | Ingestão streaming (validação do contrato) | DLQ / descarte com registro |
| Violação estrutural (presente sem nota; ausente com nota) | Silver, nesta seção | Quarentena, com motivo |
| Nulo com significado (ausente sem proficiência; resultado sem meta) | Silver, seções 4 e 5 | Permanece, com semântica documentada |

In [19]:
# --- 7.1 Aplicar as regras estruturais e separar a quarentena (D-011) ---
regras_quarentena = {
    "presente_sem_nota": df_alunos["presente"] & df_alunos["proficiencia"].isna(),
    "ausente_com_nota":  ~df_alunos["presente"] & df_alunos["proficiencia"].notna(),
}

df_alunos["motivo_quarentena"] = pd.NA
for motivo, condicao in regras_quarentena.items():
    df_alunos.loc[condicao & df_alunos["motivo_quarentena"].isna(),
                  "motivo_quarentena"] = motivo

df_quarentena = df_alunos[df_alunos["motivo_quarentena"].notna()].copy()
df_alunos_silver = (df_alunos[df_alunos["motivo_quarentena"].isna()]
                    .drop(columns="motivo_quarentena"))

print(f"Alunos avaliados pelas regras: {len(df_alunos):,}")
print(f"Aprovados para a Silver:       {len(df_alunos_silver):,}")
print(f"Quarentena:                    {len(df_quarentena):,}")
print()
print("Motivos da quarentena:")
print(df_quarentena["motivo_quarentena"].value_counts().to_string())
print()

# Conferência de conservação: nenhuma linha some no processo
conserva = len(df_alunos_silver) + len(df_quarentena) == len(df_alunos)
print(f"Conservação (aprovados + quarentena = total): {conserva}  (esperado: True)")

Alunos avaliados pelas regras: 3,867,999
Aprovados para a Silver:       3,866,814
Quarentena:                    1,185

Motivos da quarentena:
motivo_quarentena
presente_sem_nota    1185

Conservação (aprovados + quarentena = total): True  (esperado: True)


In [20]:
# --- 7.2 Auditar os nulos legítimos que permanecem na Silver ---
# Nulos que carregam significado passam pelas regras; a auditoria os conta
# e confere que cada um corresponde a uma categoria documentada.

print("Nulos legítimos preservados, por categoria:")
print()

ausentes = (~df_alunos_silver["presente"]).sum()
print(f"  Alunos ausentes sem proficiência (D-011):        {ausentes:>10,}")

rede5_s = df_municipio_silver[df_municipio_silver["rede"].astype(str) == "5"]
sem_meta_2023 = (rede5_s["ano"].eq(2023) & rede5_s["meta_taxa"].isna()).sum()
sem_meta_2024 = (rede5_s["ano"].eq(2024) & rede5_s["meta_taxa"].isna()).sum()
print(f"  Resultados 2023 sem meta (pactuação em 2024):    {sem_meta_2023:>10,}")
print(f"  Resultados 2024 sem meta na safra vigente D-013: {sem_meta_2024:>10,}")

fora_rede5 = df_municipio_silver["rede"].astype(str) != "5"
meta_na_fora = df_municipio_silver.loc[fora_rede5, "meta_taxa"].isna().sum()
print(f"  Meta nula fora da rede Pública (D-010):          {meta_na_fora:>10,}")

sem_meta_2025 = df_estimativa_2025["meta_taxa"].isna().sum()
print(f"  Estimativas 2025 sem meta pactuada (D-013):      {sem_meta_2025:>10,}")
print()

# Conferência: fora da rede Pública, nenhuma meta deve estar preenchida
vazamento = df_municipio_silver.loc[fora_rede5, "meta_taxa"].notna().sum()
print(f"Metas preenchidas fora da rede Pública: {vazamento}  (esperado: 0)")

Nulos legítimos preservados, por categoria:

  Alunos ausentes sem proficiência (D-011):           512,153
  Resultados 2023 sem meta (pactuação em 2024):         4,950
  Resultados 2024 sem meta na safra vigente D-013:        284
  Meta nula fora da rede Pública (D-010):              13,529
  Estimativas 2025 sem meta pactuada (D-013):              23

Metas preenchidas fora da rede Pública: 0  (esperado: 0)


## 8. Gravação na Silver e ensaio da produção

**Passos desta seção:** (8.1) gravar as tabelas validadas na área `silver/` do lake e a quarentena em área própria; (8.2) reconciliar cada gravação lendo o dado de volta; (8.3) registrar o plano de promoção para produção.

🎓 **Conceito, gravar uma vez, no final** (*ETL Pipelines, Aula 1*): nenhuma célula anterior escreveu no lake; a escrita acontece apenas agora, com tudo validado. Se qualquer conferência das seções 2 a 7 tivesse falhado, a Silver permaneceria intocada: é a versão em camadas do princípio de transação (tudo ou nada).

🎓 **Conceito, idempotência por sobrescrita de partição** (mesmo padrão do `prod_01`): cada tabela é gravada em `silver/{tabela}/data_processamento=AAAA-MM-DD/`. Reexecutar no mesmo dia substitui a partição do dia, sem duplicar; partições de dias diferentes formam o histórico de processamentos.

📌 **O que a Silver entrega** (e o que ela deliberadamente não entrega): `alunos` (microdados aprovados, com a flag `presente`), `municipio` (resultados integrados com dimensão e meta vigente), `estimativa_2025` (agregação do fluxo, com origem explícita), as três tabelas de metas no formato long (todas as safras, preservando o histórico de renegociação) e a quarentena. A tabela `uf` decodificada na Seção 2 foi laboratório, não entrega: os recortes por UF e região saem da tabela municipal integrada, que carrega ambos; incluí-la duplicaria o mesmo dado em outro grão sem demanda que o justifique (o mesmo princípio anti-sobre-engenharia da D-012).

In [21]:
# --- 8.1 Gravar as tabelas validadas no lake ---
def gravar_silver(df: pd.DataFrame, tabela: str, area: str = "silver") -> dict:
    """Grava uma tabela em silver/ (ou quarentena/), particionada por data."""
    garantir_credencial()
    momento = datetime.now(timezone.utc)
    df = df.copy()
    df["_processing_timestamp"] = momento.isoformat()
    destino = (f"gs://{BUCKET_LAKE}/{area}/{tabela}/"
               f"data_processamento={momento:%Y-%m-%d}/{tabela}.parquet")
    df.to_parquet(destino, index=False, storage_options={"token": credenciais})
    return {"area": area, "tabela": tabela, "linhas": len(df), "destino": destino}

entregas = [
    gravar_silver(df_alunos_silver,        "alunos"),
    gravar_silver(df_municipio_silver,     "municipio"),
    gravar_silver(df_estimativa_2025,      "estimativa_2025"),
    gravar_silver(metas_long["brasil"],    "metas_brasil"),
    gravar_silver(metas_long["uf"],        "metas_uf"),
    gravar_silver(metas_long["municipio"], "metas_municipio"),
    gravar_silver(df_quarentena,           "alunos", area="quarentena"),
]

for e in entregas:
    print(f"  {e['linhas']:>10,} linhas  ->  {e['area']}/{e['tabela']}")
print()
print(f"Tabelas gravadas: {len(entregas)}")

   3,866,814 linhas  ->  silver/alunos
      23,995 linhas  ->  silver/municipio
       1,163 linhas  ->  silver/estimativa_2025
          21 linhas  ->  silver/metas_brasil
         567 linhas  ->  silver/metas_uf
      74,928 linhas  ->  silver/metas_municipio
       1,185 linhas  ->  quarentena/alunos

Tabelas gravadas: 7


In [22]:
# --- 8.2 Reconciliação: ler de volta e conferir as contagens ---
# O mesmo princípio da ingestão batch: gravar não basta, é preciso
# provar que o que está no lake é o que saiu da memória.

print("Reconciliação gravado x relido:")
print()
divergencias = 0
for e in entregas:
    relido = pd.read_parquet(e["destino"],
                             columns=["_processing_timestamp"],
                             storage_options={"token": credenciais})
    status = "OK" if len(relido) == e["linhas"] else "DIVERGIU"
    divergencias += status != "OK"
    print(f"  {e['area']}/{e['tabela']:<18} "
          f"gravado {e['linhas']:>10,} | relido {len(relido):>10,}  {status}")

print()
print(f"Status final: {'FALHA' if divergencias else 'SUCESSO'} "
      f"({len(entregas) - divergencias} de {len(entregas)} OK)")

Reconciliação gravado x relido:

  silver/alunos             gravado  3,866,814 | relido  3,866,814  OK
  silver/municipio          gravado     23,995 | relido     23,995  OK
  silver/estimativa_2025    gravado      1,163 | relido      1,163  OK
  silver/metas_brasil       gravado         21 | relido         21  OK
  silver/metas_uf           gravado        567 | relido        567  OK
  silver/metas_municipio    gravado     74,928 | relido     74,928  OK
  quarentena/alunos             gravado      1,185 | relido      1,185  OK

Status final: SUCESSO (7 de 7 OK)


### 8.3 Da validação à produção

O ciclo validado neste notebook será promovido para `src/transform/prod_03_bronze_to_silver.py`, seguindo a convenção D-008. O que muda na promoção:

- as células viram **funções nomeadas** (decodificação, unpivot, integração, estimativa do fluxo, qualidade, gravação), orquestradas por um `main()` com relatório de execução no padrão dos scripts anteriores;
- os **erros ganham tratamento educado**: a leitura de uma tabela ausente na Bronze, que no desenvolvimento estourou um `IndexError` seco (Seção 5, quando o diretório ainda não havia sido ingerido), passa a produzir mensagem orientando a executar a ingestão;
- as **conferências viram verificações executáveis**: joins sem explosão, conservação de linhas na quarentena e reconciliação da gravação derrubam a execução com código de saída 1 em caso de violação, permitindo que a orquestração detecte a falha.

As evidências de execução permanecem neste notebook; o script de produção referencia este desenvolvimento no cabeçalho.